# ASHPB Optimization Trajectory Visualization

This notebook visualizes the iteration trajectories of optimization algorithms in 2D variable space.
It helps identify where optimizations get stuck and compare successful vs failed trajectories.


In [1]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from tqdm import tqdm
import sys
sys.path.append('src')

import enex_analysis as enex
import dartwork_mpl as dm


In [7]:
# Initialize model
ashpb_model = enex.AirSourceHeatPumpBoiler(
    ref='R410A',
    V_disp_cmp=0.002,
    eta_cmp_isen=0.7,
    UA_cond_design=1000.0,
    UA_evap_design=1000.0,
    A_cross_ou=np.pi * 0.4 ** 2,
    dV_ou_design=2.0,
    dP_ou_design=500.0,
    eta_fan_ou=0.8,
    T0=0.0,
    T_tank_w_setpoint=65.0,
    T_tank_w_lower_bound=55.0,
    T_serv_w=40.0,
    T_sup_w=15.0,
    heater_capacity=8000.0,
    dV_w_serv_m3s=0.0001,
    r0=0.2,
    H=0.8,
    x_shell=0.01,
    x_ins=0.05,
    k_shell=25,
    k_ins=0.03,
    h_o=15,
)


## Test Cases (Including Challenging Cases)

Define test cases, including some that are likely to fail or converge slowly.


In [8]:
# Test cases - including challenging ones
test_cases = [
    {'name': 'Normal', 'T_tank_w': 55.0, 'T_oa': 20.0, 'Q_cond_load': 8000.0},
    {'name': 'Challenging: Low Temp High Load', 'T_tank_w': 50.0, 'T_oa': 5.0, 'Q_cond_load': 15000.0},
    {'name': 'Challenging: Extreme Load', 'T_tank_w': 55.0, 'T_oa': 15.0, 'Q_cond_load': 20000.0},
    {'name': 'Edge: Boundary', 'T_tank_w': 55.0, 'T_oa': 20.0, 'Q_cond_load': 5000.0},
]

# Algorithm to test (can be changed)
algorithm = 'SLSQP'  # Try 'trust-constr' or 'COBYLA' as well


## Generate Feasible Region Grid

Create a grid to visualize constraint boundaries and feasible regions.


In [9]:
def evaluate_constraints_grid(model, T_tank_w, Q_cond_load, T_oa, 
                              dT_ref_cond_range, dT_ref_evap_range, tolerance=0.05):
    """Evaluate constraints on a grid"""
    cond_low_grid = np.full((len(dT_ref_cond_range), len(dT_ref_evap_range)), np.nan)
    cond_high_grid = np.full((len(dT_ref_cond_range), len(dT_ref_evap_range)), np.nan)
    evap_low_grid = np.full((len(dT_ref_cond_range), len(dT_ref_evap_range)), np.nan)
    evap_high_grid = np.full((len(dT_ref_cond_range), len(dT_ref_evap_range)), np.nan)
    feasible_grid = np.full((len(dT_ref_cond_range), len(dT_ref_evap_range)), np.nan)
    
    for i, dT_cond in enumerate(tqdm(dT_ref_cond_range, desc="Evaluating constraints")):
        for j, dT_evap in enumerate(dT_ref_evap_range):
            try:
                perf = model._calc_on_state(
                    optimization_vars=[dT_cond, dT_evap],
                    T_tank_w=T_tank_w,
                    Q_cond_load=Q_cond_load,
                    T_oa=T_oa
                )
                if perf is not None and isinstance(perf, dict):
                    Q_LMTD_cond = perf.get('Q_LMTD_cond [W]', np.nan)
                    Q_LMTD_evap = perf.get('Q_LMTD_evap [W]', np.nan)
                    Q_ref_evap = perf.get('Q_ref_evap [W]', np.nan)
                    
                    if not (np.isnan(Q_LMTD_cond) or np.isnan(Q_LMTD_evap) or np.isnan(Q_ref_evap)):
                        # Condenser constraints
                        cond_low_grid[i, j] = Q_LMTD_cond - Q_cond_load
                        cond_high_grid[i, j] = Q_cond_load * (1 + tolerance) - Q_LMTD_cond
                        
                        # Evaporator constraints
                        evap_low_grid[i, j] = Q_LMTD_evap - Q_ref_evap * (1 - tolerance)
                        evap_high_grid[i, j] = Q_ref_evap * (1 + tolerance) - Q_LMTD_evap
                        
                        # Feasible if all constraints satisfied
                        if (cond_low_grid[i, j] >= 0 and cond_high_grid[i, j] >= 0 and
                            evap_low_grid[i, j] >= 0 and evap_high_grid[i, j] >= 0):
                            feasible_grid[i, j] = 1.0
                        else:
                            feasible_grid[i, j] = 0.0
            except:
                pass
    
    return cond_low_grid, cond_high_grid, evap_low_grid, evap_high_grid, feasible_grid


## Run Optimizations and Collect Trajectories


In [10]:
# Store trajectory results
trajectory_results = {}

for case in test_cases:
    case_name = case['name']
    print(f"\nRunning optimization for {case_name}...")
    
    try:
        # Run optimization with callback
        opt_result = ashpb_model._optimize_operation(
            T_tank_w=case['T_tank_w'],
            Q_cond_load=case['Q_cond_load'],
            T_oa=case['T_oa'],
            method=algorithm,
            callback=None
        )
        
        # Extract trajectory
        history = getattr(opt_result, 'iteration_history', [])
        
        trajectory_results[case_name] = {
            'success': opt_result.success,
            'history': history,
            'final_x': opt_result.x if opt_result.success else None,
            'message': opt_result.message,
            'case': case
        }
        
        print(f"  Success: {opt_result.success}, Iterations: {len(history)}")
    except Exception as e:
        trajectory_results[case_name] = {
            'success': False,
            'history': [],
            'final_x': None,
            'message': str(e),
            'case': case
        }
        print(f"  Error: {str(e)}")



Running optimization for Normal...
  Success: True, Iterations: 0

Running optimization for Challenging: Low Temp High Load...
  Success: True, Iterations: 0

Running optimization for Challenging: Extreme Load...
  Success: True, Iterations: 0

Running optimization for Edge: Boundary...
  Success: False, Iterations: 0


## Visualize Trajectories

Plot iteration trajectories overlaid on feasible region maps.


In [6]:
# Create grid for feasible region visualization
dT_ref_cond_range = np.linspace(1.0, 30.0, 50)
dT_ref_evap_range = np.linspace(1.0, 30.0, 50)
dT_ref_cond_grid, dT_ref_evap_grid = np.meshgrid(dT_ref_cond_range, dT_ref_evap_range)

# Create subplots for each test case
n_cases = len(test_cases)
fig, axes = plt.subplots(n_cases, 1, figsize=(dm.cm2in(16), dm.cm2in(6*n_cases)))

if n_cases == 1:
    axes = [axes]

for idx, case in enumerate(test_cases):
    case_name = case['name']
    ax = axes[idx]
    
    # Evaluate constraints for this case
    cond_low, cond_high, evap_low, evap_high, feasible = evaluate_constraints_grid(
        ashpb_model, case['T_tank_w'], case['Q_cond_load'], case['T_oa'],
        dT_ref_cond_range, dT_ref_evap_range
    )
    
    # Plot feasible region
    feasible_plot = feasible.copy()
    feasible_plot[feasible_plot == 0] = np.nan
    im = ax.imshow(feasible_plot, aspect='auto', origin='lower',
                   extent=[dT_ref_evap_range[0], dT_ref_evap_range[-1],
                          dT_ref_cond_range[0], dT_ref_cond_range[-1]],
                   cmap='RdYlGn', alpha=0.3, vmin=0, vmax=1)
    
    # Plot constraint boundaries (contours at 0)
    ax.contour(dT_ref_evap_grid, dT_ref_cond_grid, cond_low, levels=[0], 
              colors='blue', linestyles='--', linewidths=1, alpha=0.5, label='Cond Low')
    ax.contour(dT_ref_evap_grid, dT_ref_cond_grid, cond_high, levels=[0], 
              colors='blue', linestyles=':', linewidths=1, alpha=0.5, label='Cond High')
    ax.contour(dT_ref_evap_grid, dT_ref_cond_grid, evap_low, levels=[0], 
              colors='red', linestyles='--', linewidths=1, alpha=0.5, label='Evap Low')
    ax.contour(dT_ref_evap_grid, dT_ref_cond_grid, evap_high, levels=[0], 
              colors='red', linestyles=':', linewidths=1, alpha=0.5, label='Evap High')
    
    # Plot trajectory
    traj_data = trajectory_results[case_name]
    history = traj_data['history']
    
    if len(history) > 0:
        # Extract trajectory points
        traj_cond = [h['dT_ref_cond'] for h in history]
        traj_evap = [h['dT_ref_evap'] for h in history]
        
        # Plot trajectory with arrows
        for i in range(len(traj_cond)):
            if i == 0:
                # Start point
                ax.plot(traj_evap[i], traj_cond[i], 'go', markersize=10, 
                       label='Start', zorder=5)
            elif i == len(traj_cond) - 1:
                # End point
                color = 'r' if not traj_data['success'] else 'b'
                marker = 'x' if not traj_data['success'] else 'o'
                label = 'End (Failed)' if not traj_data['success'] else 'End (Success)'
                ax.plot(traj_evap[i], traj_cond[i], color+marker, markersize=12,
                       label=label, zorder=5, markeredgewidth=2)
            else:
                # Intermediate points
                ax.plot(traj_evap[i], traj_cond[i], 'ko', markersize=4, 
                       alpha=0.5, zorder=3)
            
            # Draw arrow to next point
            if i < len(traj_cond) - 1:
                dx = traj_evap[i+1] - traj_evap[i]
                dy = traj_cond[i+1] - traj_cond[i]
                ax.arrow(traj_evap[i], traj_cond[i], dx, dy,
                        head_width=0.5, head_length=0.3, fc='k', ec='k',
                        alpha=0.4, zorder=2, length_includes_head=True)
            
            # Add iteration number label (every 5th iteration or last)
            if i % 5 == 0 or i == len(traj_cond) - 1:
                ax.text(traj_evap[i], traj_cond[i], f'  {i}', 
                       fontsize=7, alpha=0.7, zorder=4)
    
    ax.set_xlabel('dT_ref_evap [K]', fontsize=10)
    ax.set_ylabel('dT_ref_cond [K]', fontsize=10)
    ax.set_title(f'{case_name} (Algorithm: {algorithm}, Success: {traj_data["success"]})', 
                fontsize=11)
    ax.legend(fontsize=8, loc='upper right')
    ax.grid(True, alpha=0.3)
    ax.set_xlim(dT_ref_evap_range[0], dT_ref_evap_range[-1])
    ax.set_ylim(dT_ref_cond_range[0], dT_ref_cond_range[-1])

plt.tight_layout()
dm.save_and_show(fig)


Evaluating constraints: 100%|██████████| 50/50 [00:01<00:00, 46.58it/s]
/tmp/ipykernel_3041232/2638284104.py:32: UserWarning: The following kwargs were not used by contour: 'label'
  ax.contour(dT_ref_evap_grid, dT_ref_cond_grid, cond_low, levels=[0],
/tmp/ipykernel_3041232/2638284104.py:34: UserWarning: The following kwargs were not used by contour: 'label'
  ax.contour(dT_ref_evap_grid, dT_ref_cond_grid, cond_high, levels=[0],
/tmp/ipykernel_3041232/2638284104.py:36: UserWarning: The following kwargs were not used by contour: 'label'
  ax.contour(dT_ref_evap_grid, dT_ref_cond_grid, evap_low, levels=[0],
/tmp/ipykernel_3041232/2638284104.py:38: UserWarning: The following kwargs were not used by contour: 'label'
  ax.contour(dT_ref_evap_grid, dT_ref_cond_grid, evap_high, levels=[0],
/tmp/ipykernel_3041232/2638284104.py:85: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argum

## Detailed Analysis: Constraint Violations Along Trajectory

Analyze constraint violations at each iteration point.


In [8]:
# Analyze constraint violations for a specific case
case_to_analyze = 'Challenging: Low Temp High Load'  # Change as needed

if case_to_analyze in trajectory_results:
    traj_data = trajectory_results[case_to_analyze]
    case = traj_data['case']
    history = traj_data['history']
    
    if len(history) > 0:
        # Evaluate constraints at each iteration
        constraint_violations = []
        
        for iter_data in history:
            try:
                perf = ashpb_model._calc_on_state(
                    optimization_vars=iter_data['x'],
                    T_tank_w=case['T_tank_w'],
                    Q_cond_load=case['Q_cond_load'],
                    T_oa=case['T_oa']
                )
                if perf is not None:
                    Q_LMTD_cond = perf.get('Q_LMTD_cond [W]', np.nan)
                    Q_LMTD_evap = perf.get('Q_LMTD_evap [W]', np.nan)
                    Q_ref_evap = perf.get('Q_ref_evap [W]', np.nan)
                    tolerance = 0.05
                    
                    violations = {
                        'iteration': iter_data['iteration'],
                        'cond_low': Q_LMTD_cond - case['Q_cond_load'],
                        'cond_high': case['Q_cond_load'] * (1 + tolerance) - Q_LMTD_cond,
                        'evap_low': Q_LMTD_evap - Q_ref_evap * (1 - tolerance),
                        'evap_high': Q_ref_evap * (1 + tolerance) - Q_LMTD_evap,
                    }
                    constraint_violations.append(violations)
            except:
                pass
        
        # Create DataFrame
        df_violations = pd.DataFrame(constraint_violations)
        
        # Plot constraint violations
        fig, axes = plt.subplots(2, 2, figsize=(dm.cm2in(16), dm.cm2in(12)))
        
        # Condenser low constraint
        axes[0, 0].plot(df_violations['iteration'], df_violations['cond_low'], 'b-o', markersize=4)
        axes[0, 0].axhline(y=0, color='r', linestyle='--', alpha=0.5)
        axes[0, 0].set_xlabel('Iteration', fontsize=9)
        axes[0, 0].set_ylabel('Cond Low Constraint', fontsize=9)
        axes[0, 0].set_title('Q_LMTD_cond - Q_cond_load >= 0', fontsize=10)
        axes[0, 0].grid(True, alpha=0.3)
        
        # Condenser high constraint
        axes[0, 1].plot(df_violations['iteration'], df_violations['cond_high'], 'b-o', markersize=4)
        axes[0, 1].axhline(y=0, color='r', linestyle='--', alpha=0.5)
        axes[0, 1].set_xlabel('Iteration', fontsize=9)
        axes[0, 1].set_ylabel('Cond High Constraint', fontsize=9)
        axes[0, 1].set_title('Q_cond_load*(1+tol) - Q_LMTD_cond >= 0', fontsize=10)
        axes[0, 1].grid(True, alpha=0.3)
        
        # Evaporator low constraint
        axes[1, 0].plot(df_violations['iteration'], df_violations['evap_low'], 'g-o', markersize=4)
        axes[1, 0].axhline(y=0, color='r', linestyle='--', alpha=0.5)
        axes[1, 0].set_xlabel('Iteration', fontsize=9)
        axes[1, 0].set_ylabel('Evap Low Constraint', fontsize=9)
        axes[1, 0].set_title('Q_LMTD_evap - Q_ref_evap*(1-tol) >= 0', fontsize=10)
        axes[1, 0].grid(True, alpha=0.3)
        
        # Evaporator high constraint
        axes[1, 1].plot(df_violations['iteration'], df_violations['evap_high'], 'g-o', markersize=4)
        axes[1, 1].axhline(y=0, color='r', linestyle='--', alpha=0.5)
        axes[1, 1].set_xlabel('Iteration', fontsize=9)
        axes[1, 1].set_ylabel('Evap High Constraint', fontsize=9)
        axes[1, 1].set_title('Q_ref_evap*(1+tol) - Q_LMTD_evap >= 0', fontsize=10)
        axes[1, 1].grid(True, alpha=0.3)
        
        plt.suptitle(f'Constraint Violations: {case_to_analyze}', fontsize=12)
        plt.tight_layout()
        dm.save_and_show(fig)
        
        print(f"\nConstraint Violation Summary for {case_to_analyze}:")
        print(df_violations.describe())
    else:
        print(f"No trajectory data available for {case_to_analyze}")
else:
    print(f"Case '{case_to_analyze}' not found in results")


No trajectory data available for Challenging: Low Temp High Load
